# Tutorial: Adding a New Model to GFMBench-API

This notebook shows how to plug **your own model** into GFMBench-API.

**Design idea**

- **You implement what your model can expose** (embeddings, LM token probs, a classification head, …).
- **Tasks evaluate what they can**: each task calls the inference methods it needs and **skips metrics** when a method returns `None` or is missing a required output.

You do **not** need to implement every method, and you do not need to know which task uses which method while wiring the adapter. Implement capabilities; run tasks afterward.

**What you will do**

1. Skim the duck-typed model API (inputs / outputs of each method).
2. Build a **random mock** model method-by-method (guided cells).
3. Run two example tasks (BEND expression zero-shot; GUE promoter with a tiny linear probe).
4. Optionally swap in a small real adapter (**NTv3 8M**) the same way.

Tasks come from `gfmbench_api`. Models and the probe helper live **in this notebook** so the pattern is easy to copy.

> Inheritance from `BaseGFMModel` is **optional**. Duck typing is enough: implement methods with the right signatures and shapes.


## 0. Setup

From the repo root:

```bash
pip install -r basic_requirements.txt
pip install transformers   # only needed for Part B (NTv3)
```

Part B uses gated weights [`InstaDeepAI/NTv3_8M_pre`](https://huggingface.co/InstaDeepAI/NTv3_8M_pre): accept the license and run `huggingface-cli login` first.

Set `ROOT_DATA_DIR` below before running anything else.


### User config — update this path

Writable directory for task data (reference genomes / datasets download on first run).

In [1]:
# >>> UPDATE THIS PATH <<<
ROOT_DATA_DIR = "/path/to/data"
ROOT_DATA_DIR = "/data/sense/common/data"
print(f"ROOT_DATA_DIR = {ROOT_DATA_DIR}")


ROOT_DATA_DIR = /data/sense/common/data


In [2]:
# SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0
#
# Third-party sources used later in this notebook (see also THIRD_PARTY_NOTICES.md):
# - https://huggingface.co/InstaDeepAI/NTv3_8M_pre — gated; accept license on HuggingFace
# - https://huggingface.co/datasets/leannmlindsey/GUE — MIT
# - https://github.com/frederikkemarin/BEND — BSD-3-Clause; source benchmark for variant_effects_expression.bed,
#   which BEND distributes via https://sid.erda.dk/share_redirect/aNQa0Oz2lY/data/variant_effects/

from __future__ import annotations

from typing import List, Optional, Sequence, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Hardcoded to match usage_examples/run_benchmark.py defaults
TASK_CONFIG = {
    "max_sequence_length": 512,
    "batch_size": 32,
    "num_workers": 0,
    "max_num_samples": None,  # full datasets
}

print(f"device={DEVICE}, max_samples={TASK_CONFIG['max_num_samples'] or 'all'}")


device=cuda, max_samples=all


## 1. Model API — capabilities

Full docstrings and edge cases: `gfmbench_api/tasks/base/base_gfm_model.py`.

Think of each method as a **capability** your model may or may not support:

| Method | Role (capability) |
|:-------|:------------------|
| `infer_sequence_to_sequence` | Per-sequence LM / embedding outputs |
| `sequence_pos_to_prob_pos` | Map a DNA base index → model output index |
| `infer_masked_sequence_to_token_probs` | Masked prediction at one position (ref vs alt letters) |
| `infer_sequence_to_labels_probs` | Supervised classification head over a sequence |
| `infer_variant_ref_sequences_to_labels_probs` | Classification over a (variant, reference) pair |
| `infer_sequence_to_regression` | Continuous / binned regression outputs |

**Rule:** implement what you can; return `None` (or `(None, None)` where the API uses a tuple) for the rest. Tasks decide which metrics are possible from what you returned.


## 2. Part A — Random mock model (guided)

We build a **tiny fake GFM** with a clear internal pipeline (all weights are random NumPy — no ML libraries beyond NumPy/PyTorch already imported):

An **MLM head** maps a latent vector → vocab logits (used for per-token probs and for masked prediction).

Each inference method is added in its own cell with an explicit **inputs → outputs** contract.


### A.0 Scaffold

| Piece | Role |
|:------|:-----|
| `PseudoTokenizer` | Maps `A/T/C/G` ↔ ids; exposes `<MASK>` |
| `embedding_layer` | `token_ids →` vectors `[vocab, H]` (lookup) |
| `model` | Black-box map `token_embeddings → latent_embeddings` (same length) |
| `aggregator` | e.g. mean-pool latents → one vector per sequence |
| `mlm_head` | latent → logits over `{A,T,C,G,<MASK>}` |

`_clf_head` / `_pair_clf_head` / `_reg_head` are optional host-side linear layers we can attach later for supervised demos — not part of this backbone.


In [3]:
class PseudoTokenizer:
    """Character-level DNA tokenizer: {A,T,C,G} + <MASK>."""

    def __init__(self):
        self.token_to_id = {"A": 0, "T": 1, "C": 2, "G": 3, "<MASK>": 4}
        self.id_to_token = {i: t for t, i in self.token_to_id.items()}
        self.mask_token = "<MASK>"
        self.mask_token_id = self.token_to_id[self.mask_token]
        self.vocab_size = len(self.token_to_id)

    def encode(self, sequence: str) -> np.ndarray:
        """string → 1-D int array of token ids (unknown bases → <MASK>)."""
        ids = [self.token_to_id.get(ch, self.mask_token_id) for ch in sequence.upper()]
        return np.asarray(ids, dtype=np.int64)


class RandomMockModel:
    """Toy GFM: tokenizer → embed → black-box model → pool / MLM head (random weights)."""

    def __init__(self, device: str = "cpu", hidden_dim: int = 32, num_classes: int = 2, num_outputs: int = 1, seed: int = 0):
        self.device = device
        self.hidden_dim = hidden_dim
        self.num_classes = num_classes
        self.num_outputs = num_outputs
        # Host-side heads (attached later / optionally) — not part of the random backbone
        self._clf_head: Optional[nn.Linear] = None          # [H] → [C]
        self._pair_clf_head: Optional[nn.Linear] = None     # [2H] → [C]  (var ‖ ref)
        self._reg_head: Optional[nn.Linear] = None          # [H] → [num_outputs]

        self.tokenizer = PseudoTokenizer()
        rng = np.random.default_rng(seed)

        # embedding_layer[token_id] → vector
        self.embedding_layer = rng.normal(
            size=(self.tokenizer.vocab_size, hidden_dim)
        ).astype(np.float32)

        # black-box "model": linear map over the hidden dim (length unchanged)
        self.model_W = (0.1 * rng.normal(size=(hidden_dim, hidden_dim))).astype(np.float32)
        self.model_b = (0.1 * rng.normal(size=(hidden_dim,))).astype(np.float32)

        # MLM head: latent → vocab logits
        self.mlm_head_W = (0.1 * rng.normal(size=(hidden_dim, self.tokenizer.vocab_size))).astype(np.float32)
        self.mlm_head_b = np.zeros(self.tokenizer.vocab_size, dtype=np.float32)

    def get_hidden_dim(self) -> int:
        return self.hidden_dim

    def eval(self):
        for head in (self._clf_head, self._pair_clf_head, self._reg_head):
            if head is not None:
                head.eval()
        return self

    def train(self, mode: bool = True):
        for head in (self._clf_head, self._pair_clf_head, self._reg_head):
            if head is not None:
                head.train(mode)
        return self

    # --- building blocks ---

    def _embed(self, token_ids: np.ndarray) -> np.ndarray:
        """token_ids [...] → embeddings [..., H]."""
        return self.embedding_layer[token_ids]

    def _model(self, token_embeddings: np.ndarray) -> np.ndarray:
        """Black-box sequence→sequence map: [L, H] → [L, H]."""
        return token_embeddings @ self.model_W + self.model_b

    def _aggregator(self, latent_embeddings: np.ndarray) -> np.ndarray:
        """Mean-pool over length → [H]."""
        return latent_embeddings.mean(axis=0)

    def _mlm_logits(self, latent_vec: np.ndarray) -> np.ndarray:
        """Latent [..., H] → vocab logits [..., V]."""
        return latent_vec @ self.mlm_head_W + self.mlm_head_b

    def _softmax(self, logits: np.ndarray) -> np.ndarray:
        z = logits - logits.max(axis=-1, keepdims=True)
        e = np.exp(z)
        return e / e.sum(axis=-1, keepdims=True)


print("Scaffold ready | vocab =", list(PseudoTokenizer().token_to_id))


Scaffold ready | vocab = ['A', 'T', 'C', 'G', '<MASK>']


### A.1 `infer_sequence_to_sequence`

**Inputs**

- `sequences`: `List[str]` — batch of DNA strings, e.g. `["ATCG", "GCTA"]`
- `conditional_input`: optional `np.ndarray` of shape `[batch, num_metadata]` (unused here; may be `None`)

**Pipeline (batched)**

```text
token_ids          = pad(tokenizer.encode(seq) for seq in batch)   # [B, L]
token_embeddings   = embedding_layer[token_ids]                    # [B, L, H]
latent_embeddings  = model(token_embeddings)                       # [B, L, H]
sequence_representative = masked_mean_pool(latent_embeddings)      # [B, H]
```

Only tokenization needs a short per-string loop; embed → model → pool run on the full batch. Pad positions are zeroed in the embeddings and excluded from the mean pool.

**Outputs** — a 3-tuple; **any slot may be `None`** if that capability is missing:

| Slot | Shape | Meaning |
|:-----|:------|:--------|
| `sequence_probs` | `[B, L]` or `None` | Per position, the probability of the **actual** token at that site — e.g. next-token prediction (autoregressive) or MLM head probability under a masked input, iterated across all tokens |
| `sequence_embeddings` | `[B, L, H]` or `None` | Per-position latent token embeddings |
| `sequence_representative` | `[B, H]` or `None` | One representative vector per sequence |

Batches are padded to `max(L)` with zeros for the array outputs; reps are still pooled on the **true** length only.


In [4]:
def infer_sequence_to_sequence(
    self, sequences: List[str], conditional_input=None
) -> Tuple[Optional[np.ndarray], Optional[np.ndarray], Optional[np.ndarray]]:
    n = len(sequences)
    lengths = [len(s) for s in sequences]
    max_len = max(lengths) if lengths else 1

    # tokenize + pad to a dense batch [B, L]  (only loop: strings → ids)
    token_ids = np.zeros((n, max_len), dtype=np.int64)
    for i, sequence in enumerate(sequences):
        ids = self.tokenizer.encode(sequence)
        token_ids[i, : len(ids)] = ids

    # vectorized pipeline on the batch
    token_embeddings = self._embed(token_ids)              # [B, L, H]
    latent_embeddings = self._model(token_embeddings)      # [B, L, H]

    # mean-pool only over real tokens (ignore pad)
    pad_mask = (np.arange(max_len)[None, :] < np.asarray(lengths)[:, None]).astype(np.float32)
    pad_mask = pad_mask[..., None]                         # [B, L, 1]
    reps = (latent_embeddings * pad_mask).sum(axis=1) / np.maximum(pad_mask.sum(axis=1), 1e-9)

    embeds = latent_embeddings * pad_mask                  # zero padded positions

    # sequence_probs left unimplemented in this demo
    return None, embeds.astype(np.float32), reps.astype(np.float32)


RandomMockModel.infer_sequence_to_sequence = infer_sequence_to_sequence
print("Attached infer_sequence_to_sequence")


Attached infer_sequence_to_sequence


### A.2 `sequence_pos_to_prob_pos`

Maps a **base-pair index in the input DNA string** to the corresponding **index in the tokenizer / model output sequence**.

Tasks often know a variant’s genomic site as a character offset in the DNA string (`pos`). After tokenization, that site may land at a different token index — e.g. BPE merges several bases into one token, or special tokens (`CLS`, `SEP`, padding) shift later positions. You need this mapping whenever you extract something at the variant site from model outputs (logits, embeddings, masked-token probs).

```text
DNA:     A  T  C  G  A  ...     ← pos is an index here
tokens:  [CLS] AT CG A ...     ← return the token index that covers `pos`
```

**This mock:** char-level tokenizer, no special tokens → DNA index == model index (identity). Return `-1` if `pos` is out of range for that sequence.

**Inputs:** `sequences: List[str]`, `pos: int` (0-based DNA index)  
**Output:** `np.ndarray` shape `[B]` — output index per sequence, or `-1` if out of range


In [5]:
def sequence_pos_to_prob_pos(self, sequences: List[str], pos: int) -> np.ndarray:
    # Identity: char-level tokenizer, no special tokens.
    return np.array(
        [pos if 0 <= pos < len(s) else -1 for s in sequences], dtype=np.int64
    )


RandomMockModel.sequence_pos_to_prob_pos = sequence_pos_to_prob_pos
print("Attached sequence_pos_to_prob_pos")


Attached sequence_pos_to_prob_pos


### A.3 `infer_masked_sequence_to_token_probs`

For each sequence, mask one DNA site, run the model, and read the MLM distribution **at that masked position**. \
Return how probable the **variant** and **reference** nucleotides are under that distribution — i.e. `P(variant | sequence with site masked)` and `P(reference | sequence with site masked)`.

**Inputs**

- `sequences`: `List[str]`
- `variant_pos`: `int` — DNA index to mask
- `variant_letters` / `reference_letters`: `List[str]` length `B` (single bases)
- `conditional_input`: optional

**Outputs:** `(variant_token_probs, reference_token_probs)` each shape `[B]`, or `(None, None)` if unsupported


In [6]:
def infer_masked_sequence_to_token_probs(
    self,
    sequences: List[str],
    variant_pos: int,
    variant_letters: List[str],
    reference_letters: List[str],
    conditional_input=None,
) -> Tuple[Optional[np.ndarray], Optional[np.ndarray]]:
    n = len(sequences)
    lengths = np.asarray([len(s) for s in sequences], dtype=np.int64)
    valid = (0 <= variant_pos) & (variant_pos < lengths)
    if not np.any(valid):
        return np.zeros(n, dtype=np.float32), np.zeros(n, dtype=np.float32)

    max_len = int(lengths.max())
    token_ids = np.zeros((n, max_len), dtype=np.int64)
    for i, sequence in enumerate(sequences):
        ids = self.tokenizer.encode(sequence)
        token_ids[i, : len(ids)] = ids

    token_ids[valid, variant_pos] = self.tokenizer.mask_token_id

    latent = self._model(self._embed(token_ids))           # [B, L, H]
    probs = self._softmax(self._mlm_logits(latent[:, variant_pos]))  # [B, V]

    def _letter_ids(letters: List[str]) -> np.ndarray:
        return np.asarray(
            [self.tokenizer.token_to_id.get(x.upper(), -1) for x in letters],
            dtype=np.int64,
        )

    var_ids, ref_ids = _letter_ids(variant_letters), _letter_ids(reference_letters)
    var_out = np.zeros(n, dtype=np.float32)
    ref_out = np.zeros(n, dtype=np.float32)
    ok_var, ok_ref = valid & (var_ids >= 0), valid & (ref_ids >= 0)
    var_out[ok_var] = probs[ok_var, var_ids[ok_var]]
    ref_out[ok_ref] = probs[ok_ref, ref_ids[ok_ref]]
    return var_out, ref_out


RandomMockModel.infer_masked_sequence_to_token_probs = infer_masked_sequence_to_token_probs
print("Attached infer_masked_sequence_to_token_probs")


Attached infer_masked_sequence_to_token_probs


### A.4 `infer_sequence_to_labels_probs`

Projects each DNA sequence to a **distribution over class labels** — e.g. “is this a promoter?”, TF binding, pathogenicity bins, etc.

Typical pattern: tokenize → embed → backbone → pool to one vector per sequence, then a linear (or small MLP) head maps that vector to `num_classes` logits and softmax turns them into probabilities.

Tasks that need supervised classification call this; if you return `None`, those metrics are skipped. The head is often trained later via linear probing / fine-tuning (see the helper below).

**This mock:** `_clf_head` is the small linear layer that maps the pooled sequence vector `[H]` → class logits `[C]` (trained later by the linear-probe helper). \
Until it is set, this method returns `None` (supervised metrics that need it are skipped).

**Inputs:** `sequences`, optional `conditional_input`  
**Output:** `[B, num_classes]` probabilities, or `None` if you have no classifier


In [7]:
def infer_sequence_to_labels_probs(
    self, sequences: List[str], conditional_input=None
) -> Optional[np.ndarray]:
    if self._clf_head is None:
        return None

    # Same backbone pipeline as A.1 (tokenizer → embed → model → pool), then CLF head
    n = len(sequences)
    lengths = [len(s) for s in sequences]
    max_len = max(lengths) if lengths else 1

    token_ids = np.zeros((n, max_len), dtype=np.int64)
    for i, sequence in enumerate(sequences):
        ids = self.tokenizer.encode(sequence)
        token_ids[i, : len(ids)] = ids

    token_embeddings = self._embed(token_ids)              # [B, L, H]
    latent_embeddings = self._model(token_embeddings)      # [B, L, H]

    pad_mask = (np.arange(max_len)[None, :] < np.asarray(lengths)[:, None]).astype(np.float32)
    pad_mask = pad_mask[..., None]                         # [B, L, 1]
    reps = (latent_embeddings * pad_mask).sum(axis=1) / np.maximum(pad_mask.sum(axis=1), 1e-9)

    with torch.no_grad():
        x = torch.from_numpy(reps.astype(np.float32)).to(self.device)
        probs = F.softmax(self._clf_head(x), dim=-1)
    return probs.cpu().numpy()


RandomMockModel.infer_sequence_to_labels_probs = infer_sequence_to_labels_probs
print("Attached infer_sequence_to_labels_probs")


Attached infer_sequence_to_labels_probs


### A.5 `infer_variant_ref_sequences_to_labels_probs`

Projects a **(variant, reference) sequence pair** to a distribution over labels — used for variant-effect / pairwise classification tasks.

Typical pattern: run the **same** tokenizer → embed → backbone → pool on each sequence, concatenate the two representatives, then a pair classifier head maps `[2H] →` class logits.

```text
variant_seq → tokenizer → embed → model → pool → [H] ─┐
                                                       ├─ concat [2H] → pair_clf_head → softmax → P(label | var, ref)
ref_seq     → tokenizer → embed → model → pool → [H] ─┘
```

**This mock:** `_pair_clf_head` is the linear layer `[2H] → [C]` (starts as `None`). Until it is set, return `None`; once attached, use the same backbone pipeline as A.1 on both inputs, then `_pair_clf_head`.

**Inputs:** `variant_sequences`, `ref_sequences` (aligned `List[str]` batches), optional `conditional_input`  
**Output:** `[B, num_classes]` probabilities, or `None` if you have no pair classifier


In [8]:
def infer_variant_ref_sequences_to_labels_probs(
    self,
    variant_sequences: List[str],
    ref_sequences: List[str],
    conditional_input=None,
) -> Optional[np.ndarray]:
    if self._pair_clf_head is None:
        return None

    # Same backbone pipeline as A.1, once per sequence list
    def _to_reps(sequences: List[str]) -> np.ndarray:
        n = len(sequences)
        lengths = [len(s) for s in sequences]
        max_len = max(lengths) if lengths else 1

        token_ids = np.zeros((n, max_len), dtype=np.int64)
        for i, sequence in enumerate(sequences):
            ids = self.tokenizer.encode(sequence)
            token_ids[i, : len(ids)] = ids

        token_embeddings = self._embed(token_ids)              # [B, L, H]
        latent_embeddings = self._model(token_embeddings)      # [B, L, H]

        pad_mask = (np.arange(max_len)[None, :] < np.asarray(lengths)[:, None]).astype(np.float32)
        pad_mask = pad_mask[..., None]                         # [B, L, 1]
        return (latent_embeddings * pad_mask).sum(axis=1) / np.maximum(pad_mask.sum(axis=1), 1e-9)

    var_reps = _to_reps(variant_sequences)                     # [B, H]
    ref_reps = _to_reps(ref_sequences)                         # [B, H]
    pair = np.concatenate([var_reps, ref_reps], axis=-1)       # [B, 2H]

    with torch.no_grad():
        x = torch.from_numpy(pair.astype(np.float32)).to(self.device)
        probs = F.softmax(self._pair_clf_head(x), dim=-1)
    return probs.cpu().numpy()


RandomMockModel.infer_variant_ref_sequences_to_labels_probs = (
    infer_variant_ref_sequences_to_labels_probs
)
print("Attached infer_variant_ref_sequences_to_labels_probs")


Attached infer_variant_ref_sequences_to_labels_probs


### A.6 `infer_sequence_to_regression`

Projects each DNA sequence to **continuous outputs** (expression level, epigenomic signal, etc.) — sequence-level `[B, num_outputs]` or, for some tasks, binned `[B, num_bins, num_outputs]`.

Typical pattern: same tokenizer → embed → backbone → pool as classification, then a regression head maps `[H] →` real-valued predictions (no softmax).

```text
sequence → tokenizer → embed → model → pool → [H]
                                            → reg_head → y_hat [num_outputs]
```

**This mock:** `_reg_head` is the linear layer `[H] → [num_outputs]` (starts as `None`). Until it is set, return `None`; once attached, run the same backbone pipeline as A.1, then `_reg_head`.

**Inputs:** `sequences`, optional `conditional_input`  
**Output:** `[B, num_outputs]` predictions (this demo), or `None` if you have no regression head


In [9]:
def infer_sequence_to_regression(
    self, sequences: List[str], conditional_input=None
) -> Optional[np.ndarray]:
    if self._reg_head is None:
        return None

    # Same backbone pipeline as A.1, then regression head (no softmax)
    n = len(sequences)
    lengths = [len(s) for s in sequences]
    max_len = max(lengths) if lengths else 1

    token_ids = np.zeros((n, max_len), dtype=np.int64)
    for i, sequence in enumerate(sequences):
        ids = self.tokenizer.encode(sequence)
        token_ids[i, : len(ids)] = ids

    token_embeddings = self._embed(token_ids)              # [B, L, H]
    latent_embeddings = self._model(token_embeddings)      # [B, L, H]

    pad_mask = (np.arange(max_len)[None, :] < np.asarray(lengths)[:, None]).astype(np.float32)
    pad_mask = pad_mask[..., None]                         # [B, L, 1]
    reps = (latent_embeddings * pad_mask).sum(axis=1) / np.maximum(pad_mask.sum(axis=1), 1e-9)

    with torch.no_grad():
        x = torch.from_numpy(reps.astype(np.float32)).to(self.device)
        preds = self._reg_head(x)
    return preds.cpu().numpy()


RandomMockModel.infer_sequence_to_regression = infer_sequence_to_regression
print("Attached infer_sequence_to_regression")


Attached infer_sequence_to_regression


### A.7 Instantiate and smoke-check shapes


In [10]:
mock = RandomMockModel(device=DEVICE, hidden_dim=32, seed=0)
toy = ["ACGTACGT", "TTTT"]

# show the pipeline on one string
seq = toy[0]
token_ids = mock.tokenizer.encode(seq)
token_embeddings = mock._embed(token_ids)
latent_embeddings = mock._model(token_embeddings)
sequence_representative = mock._aggregator(latent_embeddings)
print("pipeline demo for", repr(seq))
print("  token_ids:", token_ids.tolist())
print("  token_embeddings:", token_embeddings.shape)
print("  latent_embeddings:", latent_embeddings.shape)
print("  sequence_representative:", sequence_representative.shape)

probs, embeds, reps = mock.infer_sequence_to_sequence(toy)
print("infer_sequence_to_sequence:",
      None if probs is None else probs.shape,
      None if embeds is None else embeds.shape,
      None if reps is None else reps.shape)
print("sequence_pos_to_prob_pos(pos=2):", mock.sequence_pos_to_prob_pos(toy, 2))
vp, rp = mock.infer_masked_sequence_to_token_probs(
    toy, variant_pos=1, variant_letters=["G", "A"], reference_letters=["C", "T"]
)
print("masked P(alt), P(ref):", vp, rp)
label_probs = mock.infer_sequence_to_labels_probs(toy)
print("label probs:", None if label_probs is None else label_probs.shape)
pair_probs = mock.infer_variant_ref_sequences_to_labels_probs(toy, toy)
reg = mock.infer_sequence_to_regression(toy)
print("variant/ref label probs:", None if pair_probs is None else pair_probs.shape)
print("regression:", None if reg is None else reg.shape)
print("RandomMockModel ready, hidden_dim =", mock.get_hidden_dim())


pipeline demo for 'ACGTACGT'
  token_ids: [0, 2, 3, 1, 0, 2, 3, 1]
  token_embeddings: (8, 32)
  latent_embeddings: (8, 32)
  sequence_representative: (32,)
infer_sequence_to_sequence: None (2, 8, 32) (2, 32)
sequence_pos_to_prob_pos(pos=2): [2 2]
masked P(alt), P(ref): [0.15745685 0.2317567 ] [0.19676034 0.18972184]
label probs: None
variant/ref label probs: None
regression: None
RandomMockModel ready, hidden_dim = 32


### Shared helper: train `_clf_head` (example: linear probe)

In A.4, `infer_sequence_to_labels_probs` needs `_clf_head` to map a pooled sequence vector `[H]` → class logits. Right now that head is still `None`, so supervised classification would be skipped.

User can  **adapt the model**  with whatever recipe prefers — full fine-tuning, LoRA / other PEFT, a custom optimizer schedule, `GFMFinetuner`, etc. \
 The only contract the task cares about is that after adaptation, your inference methods return the right shapes (here: `infer_sequence_to_labels_probs` → `[B, num_classes]`).

Below we show one example — **linear probing**, with hyperparameters hardcoded to match `usage_examples/run_benchmark.py` defaults (`seed=0`, `3` epochs, batch `32`, `AdamW`, `lr=3e-5`, `weight_decay=0.01`):

1. Keep the backbone **frozen** (tokenizer → embed → model → pool stay as-is).
2. For each training sequence, take only the pooled representative `[H]`.
3. Train a small `nn.Linear(H → num_classes)` on those vectors with the dataset labels (`num_classes` is an argument to the helper below; pass a **training dataset**, not the task).
4. Store the trained layer as `model._clf_head`.

After that, A.4’s method can return real `P(label | sequence)` for evaluation.


Also defines `set_seed()` (hardcoded `seed=0`, same idea as `run_benchmark.set_seed`) — call it before **each** task, including zero-shot.

**Requires:** `get_hidden_dim()`, non-`None` `sequence_representative` from `infer_sequence_to_sequence`, and `infer_sequence_to_labels_probs` reading `model._clf_head`.


In [ ]:
import os
import random


def set_seed():
    """Hardcoded seed=0"""
    seed = 0
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(True, warn_only=False)
    except AttributeError:
        try:
            torch.set_deterministic(True)
        except AttributeError:
            pass
    os.environ["PYTHONHASHSEED"] = str(seed)


def _embed_reps(model, sequences: List[str]) -> torch.Tensor:
    """Get sequence representatives as a float32 tensor on model.device."""
    _, _, reps = model.infer_sequence_to_sequence(sequences)
    if reps is None:
        raise RuntimeError("Need sequence_representative for linear probing")
    return torch.from_numpy(np.asarray(reps, dtype=np.float32)).to(model.device)


def linear_probe(model, train_dataset, num_classes: int):
    """Train a linear head on a training dataset; attach as model._clf_head.

    Hyperparameters are hardcoded to match usage_examples/run_benchmark.py defaults
    for --linear_prob (via GFMFinetuner training_params).
    """
    # optimization hyperparameters ---
    num_epochs = 3
    batch_size = 32
    num_workers = 0
    lr = 3e-5
    weight_decay = 0.01
    # ---------------------------------------------

    set_seed()

    g = torch.Generator()
    g.manual_seed(0)
    loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        generator=g,
    )

    head = nn.Linear(model.get_hidden_dim(), num_classes).to(model.device)
    opt = torch.optim.AdamW(head.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.CrossEntropyLoss()

    head.train()
    for epoch in range(num_epochs):
        total, n = 0.0, 0
        for sequences, labels, _cond in loader:
            labels = labels.to(model.device).long()
            with torch.no_grad():
                reps = _embed_reps(model, list(sequences))
            logits = head(reps)
            loss = loss_fn(logits, labels)
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.item() * labels.size(0)
            n += labels.size(0)
        print(f"  probe epoch {epoch + 1}/{num_epochs}  loss={total / max(n, 1):.4f}")

    head.eval()
    model._clf_head = head
    return head


print("linear_probe ready")


set_seed + linear_probe ready (run_benchmark.py defaults hardcoded)


### A.8 Example evaluation (mock)

Run any tasks you care about. Each task will call the methods it needs and skip what your model does not provide. A random model should look near chance until you plug in real weights (and, for supervised tasks, a trained head).

Two patterns below: **zero-shot** (evaluate only) and **supervised** (get train set → adapt → evaluate).


### Zero-shot Task → evaluate

A zero-shot GFMBench-API **task** owns the evaluation protocol (no training split / adaptation):

1. **Construct the task** (it knows its test data and which metrics to compute).
2. **`set_seed()`** (same as `run_benchmark` before each task).
3. **Evaluate** with `task.eval_test_set(model)` — it runs the held-out test split, calls your inference methods, and reports metrics (skipping any that need capabilities you returned as `None`).

So the task is the scorer here.


In [ ]:
from gfmbench_api.tasks.concrete.bend_vep_expression_task import BendVEPExpression

# 1) construct the zero-shot task
bend_task = BendVEPExpression(root_data_dir_path=ROOT_DATA_DIR, task_config=TASK_CONFIG)
print("Task:", bend_task.get_task_name())
print("Attributes:", bend_task.get_task_attributes())

# 2) seed
set_seed()

# 3) evaluate via the task (no adaptation)
mock.eval()
bend_scores_mock = bend_task.eval_test_set(mock)
print("BEND scores (RandomMock):")
for k, v in bend_scores_mock.items():
    print(f"  {k}: {v}")


/opt/conda/envs/sense-env-att/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Task: bend_variant_effects_expression
Attributes: {'has_finetuning_data': False, 'has_validation_data': False, 'is_variant_effect_prediction': True, 'is_snv_only_variants': True, 'conditional_input_metadata': None}


Evaluating (Zero-Shot): 100%|██████████| 3290/3290 [00:38<00:00, 85.00it/s]


BEND scores (RandomMock):
  sum_probs_llr_auroc: None
  sum_probs_llr_auprc: None
  sequence_embeddings_cosinesim_auroc: 0.50335805571769
  sequence_embeddings_cosinesim_auprc: 0.07515361865484244
  sequence_embeddings_l2_auroc: 0.49081164818715217
  sequence_embeddings_l2_auprc: 0.07271986936476288
  snv_variant_effect_cosinesim_auroc: 0.4800222294669737
  snv_variant_effect_cosinesim_auprc: 0.07195327511873176
  snv_variant_effect_prediction_masked_llr_auroc: 0.5099276306492818
  snv_variant_effect_prediction_masked_llr_auprc: 0.07816875270098726


### Supervised Task → train dataset → adapt → evaluate

A Supervised GFMBench-API **task** owns both the labeled data and the evaluation protocol. For a supervised example:

1. **Get the training set** from the task: `train_ds = task.get_finetune_dataset()` (sequences + labels).
2. **`set_seed()`** before adaptation / eval (mirrors `run_benchmark`).
3. **Adapt the model** with your chosen recipe — pass the **dataset** into the trainer (e.g. `linear_probe(model, train_ds, num_classes=…)`).
4. **Evaluate** with the same task: `task.eval_test_set(model)` runs the held-out test split, calls your inference methods, and reports metrics.

So the task is the source of data and the scorer.


In [ ]:
from gfmbench_api.tasks.concrete.gue_promoter_all_task import GuePromoterAllTask

gue_task = GuePromoterAllTask(root_data_dir_path=ROOT_DATA_DIR, task_config=TASK_CONFIG)
print("Task:", gue_task.get_task_name())
print("Attributes:", gue_task.get_task_attributes())

# 1) training data from the task
train_ds = gue_task.get_finetune_dataset()

# 2) seed before adapt + eval
set_seed()

# 3) adapt on that dataset (linear probe is just an example)
print("Linear probing RandomMock on GUE train split...")
linear_probe(mock, train_ds, num_classes=2)

# 4) evaluate via the task
mock.eval()
gue_scores_mock = gue_task.eval_test_set(mock)
print("GUE scores (RandomMock + probe):")
for k, v in gue_scores_mock.items():
    print(f"  {k}: {v}")


Task: gue_promoter_all
Attributes: {'has_finetuning_data': True, 'has_validation_data': True, 'task_type': 'classification', 'classification_mode': 'single_label', 'is_variant_effect_prediction': False, 'num_labels': 1, 'num_classes': 2, 'conditional_input_metadata': None}
Linear probing RandomMock on GUE train split...
  probe epoch 1/3  loss=0.6841
  probe epoch 2/3  loss=0.6779
  probe epoch 3/3  loss=0.6725


Evaluating: 100%|██████████| 185/185 [00:00<00:00, 210.19it/s]

GUE scores (RandomMock + probe):
  classification_accuracy: 0.6282094594594595
  classification_mcc: 0.25646274136055286
  classification_auroc: 0.6824735766789938
  classification_auprc: 0.7648886359322837


## 3. Part B — Real model sketch (Nucleotide Transformer v3, 8M)

Same idea as the mock: implement the capabilities your checkpoint supports; leave the rest as `None`.

We use [`InstaDeepAI/NTv3_8M_pre`](https://huggingface.co/InstaDeepAI/NTv3_8M_pre) (gated — accept the license and log in first). The backbone stays frozen; only an optional probe head trains.

NTv3-specific details worth copying if you use this checkpoint:

- Pad token length to a multiple of **128**.
- Tokenize with `add_special_tokens=False` so token index == nucleotide index (identity `sequence_pos_to_prob_pos`).

Method contracts are the same as Part A — below we only give a short reminder per cell.


### B.0 Scaffold

Load the HF checkpoint and keep helpers (`_encode`, `_forward_hidden`, `_mean_pool`). Inference methods are attached in the next cells.


In [14]:
from transformers import AutoModelForMaskedLM, AutoTokenizer

NTV3_SEQ_MULTIPLE = 128


class SimpleNTv3Model:
    """Tutorial adapter: expose embeddings / mask probs / optional clf head."""

    HF_NAME = "InstaDeepAI/NTv3_8M_pre"

    def __init__(self, device: str = "cpu", max_length: int = 1024):
        self.device = device
        self.max_length = max_length
        self._clf_head: Optional[nn.Linear] = None

        print(f"Loading {self.HF_NAME} ...")
        self.tokenizer = AutoTokenizer.from_pretrained(self.HF_NAME, trust_remote_code=True)
        self.model = AutoModelForMaskedLM.from_pretrained(self.HF_NAME, trust_remote_code=True)
        self.model.to(device).eval()
        for p in self.model.parameters():
            p.requires_grad_(False)

        self.hidden_dim = int(self.model.core.config.embed_dim)
        print(f"Loaded. hidden_dim={self.hidden_dim}")

    def get_hidden_dim(self) -> int:
        return self.hidden_dim

    def eval(self):
        self.model.eval()
        if self._clf_head is not None:
            self._clf_head.eval()
        return self

    def train(self, mode: bool = True):
        if self._clf_head is not None:
            self._clf_head.train(mode)
        return self

    def _encode(self, sequences: Sequence[str]):
        enc = self.tokenizer(
            list(sequences),
            add_special_tokens=False,
            padding=True,
            pad_to_multiple_of=NTV3_SEQ_MULTIPLE,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )
        input_ids = enc["input_ids"].to(self.device)
        attention_mask = enc.get("attention_mask", torch.ones_like(input_ids)).to(self.device)
        return input_ids, attention_mask

    def _forward_hidden(self, input_ids, attention_mask):
        return self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
        )

    def _mean_pool(self, hidden, attention_mask):
        mask = attention_mask.float().unsqueeze(-1).expand(hidden.size())
        return (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)


print("SimpleNTv3Model scaffold ready")


SimpleNTv3Model scaffold ready


### B.1 `infer_sequence_to_sequence`

Same contract as A.1. Here: last-layer hidden states as embeddings + mean-pooled reps; per-base LM probs left `None`.


In [15]:
def infer_sequence_to_sequence(
    self, sequences: List[str], conditional_input=None
) -> Tuple[Optional[np.ndarray], Optional[np.ndarray], Optional[np.ndarray]]:
    input_ids, attention_mask = self._encode(sequences)
    with torch.no_grad():
        hidden = self._forward_hidden(input_ids, attention_mask)["hidden_states"][-1]
        reps = self._mean_pool(hidden, attention_mask)
    return None, hidden.float().cpu().numpy(), reps.float().cpu().numpy()


SimpleNTv3Model.infer_sequence_to_sequence = infer_sequence_to_sequence
print("Attached infer_sequence_to_sequence")


Attached infer_sequence_to_sequence


### B.2 `sequence_pos_to_prob_pos`

Same as A.2. With `add_special_tokens=False`, DNA index == token index → identity map.


In [16]:
def sequence_pos_to_prob_pos(self, sequences: List[str], pos: int) -> np.ndarray:
    return np.full(len(sequences), pos, dtype=np.int64)


SimpleNTv3Model.sequence_pos_to_prob_pos = sequence_pos_to_prob_pos
print("Attached sequence_pos_to_prob_pos")


Attached sequence_pos_to_prob_pos


### B.3 `infer_masked_sequence_to_token_probs`

Same as A.3: mask `variant_pos`, read MLM probs for variant / reference letters at that site.


In [17]:
def infer_masked_sequence_to_token_probs(
    self,
    sequences: List[str],
    variant_pos: int,
    variant_letters: List[str],
    reference_letters: List[str],
    conditional_input=None,
) -> Tuple[Optional[np.ndarray], Optional[np.ndarray]]:
    input_ids, attention_mask = self._encode(sequences)
    if variant_pos >= input_ids.shape[1]:
        return None, None

    masked = input_ids.clone()
    masked[:, variant_pos] = self.tokenizer.mask_token_id

    with torch.no_grad():
        logits = self._forward_hidden(masked, attention_mask)["logits"].float()
        probs = torch.softmax(logits, dim=-1)

    var_p, ref_p = [], []
    unk = self.tokenizer.unk_token_id
    for i in range(len(sequences)):
        vid = self.tokenizer.convert_tokens_to_ids(variant_letters[i].upper())
        rid = self.tokenizer.convert_tokens_to_ids(reference_letters[i].upper())
        if vid == unk or rid == unk:
            var_p.append(0.0)
            ref_p.append(0.0)
            continue
        var_p.append(probs[i, variant_pos, vid].item())
        ref_p.append(probs[i, variant_pos, rid].item())
    return np.asarray(var_p, dtype=np.float32), np.asarray(ref_p, dtype=np.float32)


SimpleNTv3Model.infer_masked_sequence_to_token_probs = infer_masked_sequence_to_token_probs
print("Attached infer_masked_sequence_to_token_probs")


Attached infer_masked_sequence_to_token_probs


### B.4 `infer_sequence_to_labels_probs`

Same as A.4: pooled rep → `_clf_head` → softmax. Returns `None` until a probe attaches the head.


In [18]:
def infer_sequence_to_labels_probs(
    self, sequences: List[str], conditional_input=None
) -> Optional[np.ndarray]:
    if self._clf_head is None:
        return None
    _, _, reps = self.infer_sequence_to_sequence(sequences)
    with torch.no_grad():
        x = torch.from_numpy(np.asarray(reps, dtype=np.float32)).to(self.device)
        return F.softmax(self._clf_head(x), dim=-1).cpu().numpy()


SimpleNTv3Model.infer_sequence_to_labels_probs = infer_sequence_to_labels_probs
print("Attached infer_sequence_to_labels_probs")


Attached infer_sequence_to_labels_probs


### B.5 `infer_variant_ref_sequences_to_labels_probs`

Same as A.5. Not supported in this sketch → `None`.


In [19]:
def infer_variant_ref_sequences_to_labels_probs(
    self, variant_sequences, ref_sequences, conditional_input=None
):
    return None


SimpleNTv3Model.infer_variant_ref_sequences_to_labels_probs = (
    infer_variant_ref_sequences_to_labels_probs
)
print("Attached infer_variant_ref_sequences_to_labels_probs")


Attached infer_variant_ref_sequences_to_labels_probs


### B.6 `infer_sequence_to_regression`

Same as A.6. Not supported in this sketch → `None`.


In [20]:
def infer_sequence_to_regression(self, sequences, conditional_input=None):
    return None


SimpleNTv3Model.infer_sequence_to_regression = infer_sequence_to_regression
print("Attached infer_sequence_to_regression")


Attached infer_sequence_to_regression


### B.7 Instantiate


In [21]:
ntv3 = SimpleNTv3Model(device=DEVICE, max_length=512)
print("NTv3 ready, hidden_dim =", ntv3.get_hidden_dim())


Loading InstaDeepAI/NTv3_8M_pre ...
Loaded. hidden_dim=256
NTv3 ready, hidden_dim = 256


### B.8 Example evaluation (NTv3)

Same zero-shot / supervised task flows as A.8.


In [ ]:
set_seed()

ntv3.eval()
bend_scores_ntv3 = bend_task.eval_test_set(ntv3)
print("BEND scores (NTv3 8M):")
for k, v in bend_scores_ntv3.items():
    print(f"  {k}: {v}")


Evaluating (Zero-Shot): 100%|██████████| 3290/3290 [04:50<00:00, 11.33it/s]


BEND scores (NTv3 8M):
  sum_probs_llr_auroc: None
  sum_probs_llr_auprc: None
  sequence_embeddings_cosinesim_auroc: 0.523335745989371
  sequence_embeddings_cosinesim_auprc: 0.0797880887836499
  sequence_embeddings_l2_auroc: 0.5488693084605598
  sequence_embeddings_l2_auprc: 0.08608048811967828
  snv_variant_effect_cosinesim_auroc: 0.45153044417293847
  snv_variant_effect_cosinesim_auprc: 0.06665718399896617
  snv_variant_effect_prediction_masked_llr_auroc: 0.48976841898905654
  snv_variant_effect_prediction_masked_llr_auprc: 0.06897397173403318


In [24]:
# Reuse the same task flow: adapt → test
set_seed()
train_ds = gue_task.get_finetune_dataset()

print("Linear probing NTv3 on GUE train split...")
linear_probe(ntv3, train_ds, num_classes=2)

ntv3.eval()
gue_scores_ntv3 = gue_task.eval_test_set(ntv3)
print("GUE scores (NTv3 8M + probe):")
for k, v in gue_scores_ntv3.items():
    print(f"  {k}: {v}")


Linear probing NTv3 on GUE train split...
  probe epoch 1/3  loss=1.2062
  probe epoch 2/3  loss=0.5421
  probe epoch 3/3  loss=0.4756


Evaluating: 100%|██████████| 185/185 [00:04<00:00, 42.54it/s]

GUE scores (NTv3 8M + probe):
  classification_accuracy: 0.8327702702702703
  classification_mcc: 0.6733017200250538
  classification_auroc: 0.9048577820389901
  classification_auprc: 0.8881139899075696


## 4. Takeaways

1. **Expose capabilities** — implement the inference methods your model truly supports, with the documented shapes.
2. **Stub the rest with `None`** — tasks skip metrics they cannot compute.
3. **Do not design the adapter around a task list** — run tasks after the adapter works; they will consume what you provided.
4. For a full multi-task suite, see `usage_examples/run_benchmark.py`. If model deps conflict with `basic_requirements.txt`, see the README (*Isolated model environments*) and `usage_examples/sanity_models/isolated_mock_model.py`.

You are ready to swap in your own weights and evaluate.
